<a href="https://colab.research.google.com/github/nadeemaadi451-droid/NL-to-SQL-Transformer-from-scrach/blob/main/NL_to_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
!pip install -q datasets
from datasets import load_dataset
ds = load_dataset("gretelai/synthetic_text_to_sql")

ds.shape


README.md:   0%|          | 0.00/8.19k [00:00<?, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…): reconstructing file:   0%|          |  0.00B / 32.4MB            

synthetic_text_to_sql_train.snappy.parqu(…): downloading bytes:           |  0.00B            

synthetic_text_to_sql_test.snappy.parque(…): reconstructing file:   0%|          |  0.00B / 1.90MB            

synthetic_text_to_sql_test.snappy.parque(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

{'train': (100000, 11), 'test': (5851, 11)}

In [5]:
for old_col, new_col in [("sql_prompt", "question"), ("sql_context", "context"), ("sql", "answer")]:
    if old_col in ds["train"].column_names:
        ds = ds.rename_column(old_col, new_col)

ds = ds.remove_columns([c for c in ds["train"].column_names
                       if c not in ["context", "question", "answer"]])

In [6]:
print(ds["train"].column_names)    # ['context', 'question', 'answer']
print(ds["train"][0])

['question', 'context', 'answer']
{'question': 'What is the total volume of timber sold by each salesperson, sorted by salesperson?', 'context': "CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');", 'answer': 'SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;'}


In [7]:
import re

ds = ds.map(lambda row: {"context": re.sub(r"INSERT INTO .*?;", "", row["context"])})

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5851 [00:00<?, ? examples/s]

In [10]:
lens = [len(c) for c in ds["train"]["context"]]
print("avg:", sum(lens) / len(lens), "| max:", max(lens))

avg: 115.17202 | max: 1117


In [9]:
import re
ds = ds.map(lambda row: {"context": re.sub(r"INSERT INTO .*?;", "", row["context"], flags=re.S | re.I)})

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5851 [00:00<?, ? examples/s]

In [11]:
import numpy as np
lens = np.array([len(c) for c in ds["train"]["context"]])
for p in [50, 90, 95, 99, 99.9]:
    print(p, "percentile:", np.percentile(lens, p))
print("rows over 600 chars:", (lens > 600).sum(), "of", len(lens))

50 percentile: 101.0
90 percentile: 175.0
95 percentile: 208.0
99 percentile: 300.0
99.9 percentile: 533.0010000000038
rows over 600 chars: 55 of 100000


In [12]:
ds = ds.filter(lambda row: len(row["context"]) <= 600)
print({k: len(v) for k, v in ds.items()})

Filter:   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5851 [00:00<?, ? examples/s]

{'train': 99945, 'test': 5848}


In [13]:
import numpy as np
for col in ["question", "answer"]:
    l = np.array([len(x) for x in ds["train"][col]])
    print(col, "| avg:", l.mean(), "| 95th:", np.percentile(l, 95), "| max:", l.max())

question | avg: 83.13708539696833 | 95th: 131.0 | max: 438
answer | avg: 127.10325679123518 | 95th: 256.0 | max: 761


In [14]:
# 10% of train becomes validation; the official test split stays untouched
split = ds["train"].train_test_split(test_size=0.1, seed=42)
train_ds, val_ds = split["train"], split["test"]
test_ds = ds["test"]                      # official test set, used once at the end

print(len(train_ds), len(val_ds), len(test_ds))

89950 9995 5848


In [17]:
SEP = "<sep>"
EOS = "<eos>"

def format_data(train_ds):
  texts = []
  for row in train_ds:
    text = row["context"] + " " + row["question"] + " <sep> " + row["answer"] + " <eos>"
    if len(text) <= 600:
        texts.append(text)
  return texts

texts = format_data(train_ds)
val_texts = format_data(val_ds)

print(len(train_ds), "->", len(texts))
print(len(val_ds), "->", len(val_texts))
print(texts[0])

89950 -> 87150
9995 -> 9685
CREATE TABLE menu_sales (item VARCHAR(255), sales INTEGER, sale_date DATE);  Which menu item sold the most in February 2022? <sep> SELECT item, MAX(sales) FROM menu_sales WHERE sale_date BETWEEN '2022-02-01' AND '2022-02-28' GROUP BY item; <eos>


In [18]:
!pip install -q tokenizers

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers
special_tokens = ["<pad>", "<bos>", "<eos>", "<sep>"]
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=5000, special_tokens=special_tokens, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
tokenizer.train_from_iterator(texts, trainer=trainer)
tokenizer.get_vocab_size()

enc = tokenizer.encode(texts[0])
print(enc.tokens)
print(enc.ids)

dec = tokenizer.decode(enc.ids)
print(dec)

import numpy as np
length = [len(tokenizer.encode(i).ids) for i in texts]
print(len(length))

['CREATE', 'ĠTABLE', 'Ġmenu', '_', 'sales', 'Ġ(', 'item', 'ĠVARCHAR', '(', '255', '),', 'Ġsales', 'ĠINTEGER', ',', 'Ġsale', '_', 'date', 'ĠDATE', ');', 'Ġ', 'ĠWhich', 'Ġmenu', 'Ġitem', 'Ġsold', 'Ġthe', 'Ġmost', 'Ġin', 'ĠF', 'ebruary', 'Ġ2022', '?', 'Ġ', '<sep>', 'ĠSELECT', 'Ġitem', ',', 'ĠMAX', '(', 'sales', ')', 'ĠFROM', 'Ġmenu', '_', 'sales', 'ĠWHERE', 'Ġsale', '_', 'date', 'ĠBETWEEN', "Ġ'", '2022', '-', '02', '-', '01', "'", 'ĠAND', "Ġ'", '2022', '-', '02', '-', '2', '8', "'", 'ĠGROUP', 'ĠBY', 'Ġitem', ';', 'Ġ', '<eos>']
[303, 306, 1176, 66, 968, 287, 876, 301, 11, 386, 318, 764, 1271, 15, 1140, 66, 383, 405, 297, 224, 817, 1176, 1266, 1466, 292, 1085, 309, 293, 4990, 801, 34, 224, 3, 330, 1266, 15, 690, 11, 968, 12, 320, 1176, 66, 968, 339, 1140, 66, 383, 792, 307, 827, 16, 2545, 16, 540, 10, 392, 307, 827, 16, 2545, 16, 21, 27, 10, 404, 366, 1266, 30, 224, 2]
CREATE TABLE menu_sales (item VARCHAR(255), sales INTEGER, sale_date DATE);  Which menu item sold the most in February 2022

In [19]:
lengths = [len(tokenizer.encode(t).ids) for t in texts]
print(max(lengths), sum(lengths) / len(lengths))

254 83.49637406769936


In [20]:
context_size = 254
batch_size = 32
num_head = 4
n_embed = 128
n_layer=3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dropout = 0.2

In [21]:
"""Encoder-decoder batching:
- encoder input  (src)    = schema + question          (everything before <sep>)
- decoder input  (dec_in) = <bos> + answer             (shifted right)
- labels         (y)      = answer + <eos>             (shifted left)
Shorter examples are padded; padded labels are -100 so the loss ignores them.
"""

import torch
Pad_id = tokenizer.token_to_id("<pad>")
Bos_id = tokenizer.token_to_id("<bos>")
Eos_id = tokenizer.token_to_id("<eos>")
Sep_id = tokenizer.token_to_id("<sep>")


In [22]:
def get_batch(data, batch_size):
    ix = torch.randint(len(data), (batch_size,))
    seqs = [tokenizer.encode(data[int(i)]).ids for i in ix]

    srcs, tgts = [], []
    for s in seqs:
        sep_pos = s.index(Sep_id)
        srcs.append(s[:sep_pos])                  # schema + question
        tgts.append([Bos_id] + s[sep_pos + 1:])   # <bos> answer <eos>

    src_len = max(len(s) for s in srcs)
    tgt_len = max(len(t) for t in tgts) - 1
     # decoder and encoder input will be different
     # encoder input will be the context+question
     # decoder input will be <bos>+ answer
     # label will be answer+ <eos>
    src = torch.full((batch_size, src_len), Pad_id)
    dec_in = torch.full((batch_size, tgt_len), Pad_id)
    y = torch.full((batch_size, tgt_len), -100)
    #print(src.shape)
    #print( dec_in.shape)
    #print(y.shape)
    for b, (s, t) in enumerate(zip(srcs, tgts)): # b is the batch row
        src[b, :len(s)] = torch.tensor(s)
        dec_in[b, :len(t) - 1] = torch.tensor(t[:-1])
        y[b, :len(t) - 1] = torch.tensor(t[1:])
    return src.to(device), dec_in.to(device), y.to(device)

src, dec_in, y = get_batch(texts, batch_size)
print(src.shape, dec_in.shape, y.shape)
print(src[0])
print(dec_in[0])
print(y[0])

torch.Size([32, 75]) torch.Size([32, 85]) torch.Size([32, 85])
tensor([ 303,  306, 2653, 3608,  287, 1526,  301,   11,  342,  318, 1041,  278,
         297,  431,  363,  361,  292,  399, 1041,  356, 2135, 3319,  482, 1520,
        3230, 4830,   17,  468,  370,  280,  453,  310, 4830,   17,   34,  224,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0], device='cuda:0')
tensor([   1,  330,  451,   11, 1093,   12,  320, 2653, 3608,  339,  939,  271,
         705, 3477, 3230, 4830,   17,  502,  307,   42,  280,  453,  310, 4830,
          17,  780,  224,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,

In [23]:
import torch.nn as nn
import torch.nn.functional as F
import math

# ---- Decoder self-attention (causal) ----
class Masked_Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias=False)
    self.query = nn.Linear(n_embed, head_size, bias=False)
    self.value = nn.Linear(n_embed, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))
    self.dropout = nn.Dropout(dropout)
  def forward(self, x):
    B, T, C = x.shape
    wei = self.query(x) @ self.key(x).transpose(-2, -1) / (self.key(x).shape[-1] ** 0.5)  # FIX: query @ key^T
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)
    out = wei @ self.value(x)
    return out

In [24]:
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias=False)
    self.query = nn.Linear(n_embed, head_size, bias=False)
    self.value = nn.Linear(n_embed, head_size, bias=False)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x, mask=None):
    B, T, C = x.shape
    wei = self.query(x) @ self.key(x).transpose(-2, -1) / (self.key(x).shape[-1] ** 0.5)
    if mask is not None:                                   # mask: (B, T), 1 = real token
      wei = wei.masked_fill(mask[:, None, :] == 0, float('-inf')) # give infinity to padded values so they don't get any probablity value during softmax
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)
    out = wei @ self.value(x)
    return out

In [25]:
class MultiHead_Masked(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Masked_Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_embed)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out


In [26]:
class MultiHead(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_embed)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x, mask=None):
    out = torch.cat([h(x, mask) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out

In [27]:
class FeedForward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embed, 4 * n_embed),
        nn.ReLU(),
        nn.Linear(4 * n_embed, n_embed),
        nn.Dropout(dropout)
    )
  def forward(self, x):
    return self.net(x)

In [28]:
class Cross_Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias=False)
    self.query = nn.Linear(n_embed, head_size, bias=False)
    self.value = nn.Linear(n_embed, head_size, bias=False)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x, enc_out, mask=None):
    q = self.query(x)
    k = self.key(enc_out)
    v = self.value(enc_out)
    wei = q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5)
    if mask is not None:                    # don't attend to encoder <pad> positions
      wei = wei.masked_fill(mask[:, None, :] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)
    return wei @ v

In [29]:
class MultiHead_Cross(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Cross_Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_embed)
    self.dropout = nn.Dropout(dropout)
  def forward(self, x, enc_out, mask=None):
    out = torch.cat([h(x, enc_out, mask) for h in self.heads], dim=-1)
    return self.dropout(self.proj(out))


In [30]:
class Encoder(nn.Module):
  def __init__(self, num_head, n_embed):
    super().__init__()
    self.sa = MultiHead(num_head, n_embed // num_head)
    self.ffwd = FeedForward(n_embed)
    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)
  def forward(self, x, mask=None):
    x = x + self.sa(self.ln1(x), mask)
    x = x + self.ffwd(self.ln2(x))
    return x

In [31]:
class Decoder(nn.Module):
  def __init__(self, num_head, n_embed):
    super().__init__()
    self.sa = MultiHead_Masked(num_head, n_embed // num_head)
    self.ca = MultiHead_Cross(num_head, n_embed // num_head)
    self.ffwd = FeedForward(n_embed)
    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)
    self.ln3 = nn.LayerNorm(n_embed)
  def forward(self, x, enc_out, src_mask=None):
    x = x + self.sa(self.ln1(x))
    x = x + self.ca(self.ln2(x), enc_out, src_mask)
    x = x + self.ffwd(self.ln3(x))
    return x

In [32]:
class Transformer(nn.Module):
    def __init__(self, num_head, n_embed):
        super().__init__()
        self.encoder = nn.ModuleList([Encoder(num_head, n_embed) for _ in range(n_layer)])
        self.decoder = nn.ModuleList([Decoder(num_head, n_embed) for _ in range(n_layer)])
        self.ln_enc = nn.LayerNorm(n_embed)
        self.ln_dec = nn.LayerNorm(n_embed)

    def forward(self, src_emb, tgt_emb, src_mask=None):
        x = src_emb
        for blk in self.encoder:
            x = blk(x, src_mask)
        enc_out = self.ln_enc(x)

        y = tgt_emb
        for blk in self.decoder:
            y = blk(y, enc_out, src_mask)
        return self.ln_dec(y)

In [36]:
class MyLLM(nn.Module):
  def __init__(self, vocab_size, n_embed, context_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, n_embed)
    self.positional_encoding = nn.Embedding(context_size, n_embed)
    self.transformer = Transformer(num_head, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def embed(self, ids):
    pos = torch.arange(ids.shape[1], device=ids.device)
    return self.embedding(ids) + self.positional_encoding(pos)

  def forward(self, src, dec_in, targets=None):
    src_mask = (src != Pad_id)
    x = self.transformer(self.embed(src), self.embed(dec_in), src_mask)
    logits = self.lm_head(x)                      # keep [B, T, C]
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T),
                             ignore_index=-100)   # don't overwrite logits
    return logits, loss

  @torch.no_grad()
  def generate(self, src, max_new_tokens):
    self.eval()
    idx = torch.full((src.shape[0], 1), Bos_id, dtype=torch.long, device=src.device)
    finished = torch.zeros(src.shape[0], dtype=torch.bool, device=src.device)
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -context_size:]
      logits, _ = self(src, idx_cond)
      logits = logits[:, -1, :]
      idx_next = logits.argmax(dim=-1, keepdim=True)   # greedy: repeatable results
      idx = torch.cat((idx, idx_next), dim=1)
      finished |= (idx_next.squeeze(1) == Eos_id)
      if finished.all():
        break
    return idx

model = MyLLM(tokenizer.get_vocab_size(), n_embed, context_size).to(device)
logits, loss = model(src, dec_in, y)
print("Logits shape:", logits.shape)   # [B, tgt_len, vocab_size]
print("Loss:", loss)

Logits shape: torch.Size([32, 75, 5000])
Loss: tensor(8.7159, device='cuda:0', grad_fn=<NllLossBackward0>)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model.to(device)

# Define missing hyperparameters and configurations
max_iters = 15000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for _ in range(max_iters):
  src, dec_in, y= get_batch(texts, batch_size)
  logits, loss= model(src, dec_in, y)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss)


In [38]:
import os, torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model.to(device)

CKPT_DIR = "/content/drive/MyDrive/sql_ckpt_v2"     # new folder, keeps v1 files safe
os.makedirs(CKPT_DIR, exist_ok=True)
LATEST = f"{CKPT_DIR}/latest.pt"     # full state, for resuming
BEST   = f"{CKPT_DIR}/best.pt"       # weights with the lowest validation loss

batch_size    = 32
max_iters     = 16000                 # ~2 epochs
eval_interval = 500                  # evaluate and checkpoint every 500 steps
eval_iters    = 20                   # 200 is slow; 20 batches is enough
learning_rate = 3e-4                 # 1e-3 is too high for this setup

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

start_step, best_val = 0, float("inf")
if os.path.exists(LATEST):           # resume automatically after a disconnect
    ck = torch.load(LATEST, map_location=device)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    start_step, best_val = ck["step"], ck["best_val"]
    print(f"resumed from step {start_step}, best val {best_val:.3f}")

@torch.no_grad()
def eval_loss(data):
    model.eval()
    ls = []
    for _ in range(eval_iters):
        s, d, t = get_batch(data, batch_size)
        _, l = model(s, d, t)
        ls.append(l.item())
    model.train()
    return sum(ls) / len(ls)

model.train()
for step in range(start_step + 1, max_iters + 1):
    src, dec_in, y = get_batch(texts, batch_size)
    logits, loss = model(src, dec_in, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step}: loss {loss.item():.3f}")

    if step % eval_interval == 0:
        tr, va = eval_loss(texts), eval_loss(val_texts)
        print(f"== step {step}: train {tr:.3f} | val {va:.3f}")
        if va < best_val:
            best_val = va
            torch.save(model.state_dict(), BEST)
            print("   new best saved")
        torch.save({"model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "step": step, "best_val": best_val}, LATEST)

print("done, best val loss:", best_val)

Using device: cuda
resumed from step 8000, best val 1.342
step 8100: loss 1.383
step 8200: loss 1.693
step 8300: loss 1.370
step 8400: loss 1.089
step 8500: loss 1.276
== step 8500: train 1.140 | val 1.292
   new best saved
step 8600: loss 1.732
step 8700: loss 1.555
step 8800: loss 1.346
step 8900: loss 1.183
step 9000: loss 1.136
== step 9000: train 1.195 | val 1.235
   new best saved
step 9100: loss 1.524
step 9200: loss 1.149
step 9300: loss 1.432
step 9400: loss 1.413
step 9500: loss 1.273
== step 9500: train 1.130 | val 1.187
   new best saved
step 9600: loss 1.167
step 9700: loss 1.437
step 9800: loss 1.292
step 9900: loss 1.405
step 10000: loss 1.081
== step 10000: train 1.132 | val 1.204
step 10100: loss 1.140
step 10200: loss 0.915
step 10300: loss 1.336
step 10400: loss 1.029
step 10500: loss 1.145
== step 10500: train 1.072 | val 1.160
   new best saved
step 10600: loss 1.492
step 10700: loss 1.178
step 10800: loss 1.269
step 10900: loss 0.917
step 11000: loss 1.302
== step

In [39]:
print(loss)

tensor(0.8596, device='cuda:0', grad_fn=<NllLossBackward0>)


In [41]:
model.load_state_dict(torch.load("/content/drive/MyDrive/sql_ckpt_v2/best.pt", map_location=device))
print("train:", eval_loss(texts))
print("val  :", eval_loss(val_texts))

train: 0.831847408413887
val  : 0.9762090831995011


In [42]:
import re

# 1. load the v2 best weights (same settings as training)
model = MyLLM(tokenizer.get_vocab_size(), n_embed, context_size).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/sql_ckpt_v2/best.pt", map_location=device))
model.eval()

# 2. which set to evaluate: validation while developing, test only at the very end
eval_texts = val_texts            # later: eval_texts = format_data(test_ds)

# 3. generate SQL for one example
def predict(text):
    ids = tokenizer.encode(text).ids
    sep_pos = ids.index(Sep_id)
    src = torch.tensor([ids[:sep_pos]], device=device)        # schema + question
    ref_ids = ids[sep_pos + 1:]
    if ref_ids and ref_ids[-1] == Eos_id:
        ref_ids = ref_ids[:-1]

    out = model.generate(src, max_new_tokens=256)[0].tolist()[1:]   # drop <bos>
    if Eos_id in out:
        out = out[:out.index(Eos_id)]
    return tokenizer.decode(out), tokenizer.decode(ref_ids)

def normalize(s):
    s = s.lower().strip().rstrip(";")
    s = re.sub(r"\s*([,()=<>*])\s*", r"\1", s)    # ignore spaces around symbols
    s = s.replace('"', "'")
    return " ".join(s.split())

# 4. look at 10 examples
for t in eval_texts[:10]:
    pred, ref = predict(t)
    print("PRED:", pred)
    print("REF :", ref)
    print("OK" if normalize(pred) == normalize(ref) else "DIFFERENT")
    print()

# 5. exact match on 200 examples
N = 200
correct, fails = 0, []
for t in eval_texts[:N]:
    pred, ref = predict(t)
    if normalize(pred) == normalize(ref):
        correct += 1
    else:
        fails.append((pred, ref))
print(f"Exact match: {correct}/{N} = {correct/N:.1%}")

# 6. a few failures to study
for pred, ref in fails[:5]:
    print("PRED:", pred)
    print("REF :", ref)
    print()

PRED:  SELECT State, SUM(Amount) FROM Donors WHERE Donation = TRUE AND Donation = TRUE GROUP BY State; 
REF :  SELECT State, SUM(Amount) as 'Total Donated by New Donors' FROM Donors WHERE IsNewDonor = true AND DonationDate BETWEEN '2021-04-01' AND '2021-06-30' GROUP BY State; 
DIFFERENT

PRED:  SELECT project_name, cost FROM Building_Infrastructure ORDER BY cost DESC; 
REF :  SELECT project_name, cost, RANK() OVER (ORDER BY cost DESC) AS project_rank FROM Building_Infrastructure; 
DIFFERENT

PRED:  SELECT name, biomass FROM fish_species ORDER BY biomass DESC LIMIT 1; 
REF :  SELECT name, MAX(biomass) as max_biomass FROM fish_species; 
DIFFERENT

PRED:  INSERT INTO irrigation_system (system_id, system_type, status) VALUES (1, 'Recycling1', 'Recycling', 'Phoming', 'Plver', 'Card'); 
REF :  INSERT INTO irrigation_system (irrigation_id, system_type, status) VALUES (1, 'drip', 'active'); 
DIFFERENT

PRED:  SELECT COUNT(*) FROM PlayerGameHistory WHERE Platform = 'VR' AND PlatformDate >= DATE

In [43]:
def to_sql(context, question, max_new_tokens=256):
    text = context + " " + question
    ids = tokenizer.encode(text).ids
    src = torch.tensor([ids], device=device)
    out = model.generate(src, max_new_tokens=max_new_tokens)[0].tolist()[1:]   # drop <bos>
    if Eos_id in out:
        out = out[:out.index(Eos_id)]                                          # cut at <eos>
    return tokenizer.decode(out)

examples = [
    # example 1: group by
    ("CREATE TABLE employees (id INTEGER, name VARCHAR, salary INTEGER, department VARCHAR)",
     "What is the average salary of employees in each department?"),

    # example 2: join
    ("CREATE TABLE customers (id INT, name VARCHAR(50), city VARCHAR(50)); "
     "CREATE TABLE orders (id INT, customer_id INT, amount INT)",
     "What is the total order amount for each customer name?"),
]

for context, question in examples:
    print("Q  :", question)
    print("SQL:", to_sql(context, question))
    print()

Q  : What is the average salary of employees in each department?
SQL:  SELECT department, AVG(salary) FROM employees GROUP BY department; 

Q  : What is the total order amount for each customer name?
SQL:  SELECT c.name, SUM(d.amount) as total_amount FROM customers c JOIN orders o ON c.id = o.customer_id GROUP BY c.name; 

